# AA IV 2026-I: Object Localization

- Camilo Arciniegas
- Alexander Torres

# 1. Setup
Importaciones, semilla y configuración de dispositivo.

In [ ]:
!pip install torchsummary

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
from __future__ import print_function, division

from tqdm.notebook import tqdm
tqdm.pandas()


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from numpy.typing import NDArray
from functools import reduce
from itertools import islice
import wandb
import math
from itertools import chain
import copy
from PIL import Image

import torch
from torch import nn
from torch import Tensor
from torch.optim import Optimizer
import torch.nn.functional as F
import torchvision 
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils
from torchsummary import summary
# Import albumentations library in order to -use pre-built augmentations
import albumentations as A

from sklearn.model_selection import train_test_split
from multiprocessing import cpu_count

import os
import torch
import os.path as osp
from skimage import io, transform
import matplotlib.pyplot as plt
import typing as ty
import cv2


plt.ion()   # interactive mode

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

for root, dirs, filenames in os.walk('/kaggle/input'):
    for i, filepath in enumerate(filenames):
        if i >= 10:
            print()
            break
        print(osp.join(root, filepath))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
torch.manual_seed(32)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {device}')
test = torch.ones((100, 100)).to(device)
del test
torch.cuda.empty_cache()

Fijamos semilla para que los resultados sean reproducibles.


# 2. Dataset & Análisis Exploratorio

In [ ]:
DATA_DIR = '/kaggle/input/competitions/aa-iv-2026-i-object-localization/'
WORK_DIR = '/kaggle/working'
BATCH_SIZE = 32

img_dir = osp.join(DATA_DIR, "cars/cars_resize")
h, w, c = 416, 416, 3 # The heigh, width and number of channels of each image

# Here we map each class to an index from 0 to n_classes - 

df = pd.read_csv(osp.join(DATA_DIR, "train.csv"))

#df.drop(columns=['class_id','class'], inplace=True)
#df.rename(columns = {'id_tipo':'class_id','Tipo':'class'}, inplace=True)

id2obj = df[['class_id','class']].drop_duplicates().reset_index().reset_index().drop(columns=['index','class_id']).rename(columns={'level_0':'class_id'}).set_index('class_id')['class'].to_dict()
obj2id = df[['class_id','class']].drop_duplicates().reset_index().reset_index().drop(columns=['index','class_id']).rename(columns={'level_0':'class_id'}).set_index('class')['class_id'].to_dict()

df["class_id"] = df["class"].map(obj2id)


In [ ]:
df.head()

In [ ]:
list_image = list(df.filename)
data_shape = []
data_dim = []
data_w = []
data_h = []

for i in list_image: ## tqdm(list_image)dura 40 segundos
    ruta_imagen = osp.join(img_dir, i)
    imagen = io.imread(ruta_imagen)
    shapes = imagen.shape
    dimen = imagen.ndim
    imagen = Image.open(ruta_imagen)
    w, h = imagen.size
    data_w.append(w)
    data_h.append(h)
    data_shape.append(shapes)
    data_dim.append(dimen)

data_w_h = pd.DataFrame([list_image,data_shape,data_dim,data_w,data_h]).T.rename(columns={0:'filename',1:'shapes',2:'ndim',3:'w',4:'h'}) 

In [ ]:
data_w_h['shapes'].value_counts()

In [ ]:
data_w_h['ndim'].value_counts()

In [ ]:
df = df.merge(data_w_h[['filename','w','h']], how='inner', on='filename')
num_cols = ['xmin','xmax','ymin','ymax','w','h','class_id']
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
df.head()

In [ ]:
df['w'].value_counts()

In [ ]:
df['h'].value_counts()

In [ ]:
df.info()

In [ ]:
df['xmin'] = round(df['xmin']/df['w'],2)
df['xmax'] = round(df['xmax']/df['w'],2)
df['ymin'] = round(df['ymin']/df['h'],2)
df['ymax'] = round(df['ymax']/df['h'],2)

columns_f = ['filename', 'xmin','ymin','xmax','ymax','class_id','class','w','h']
df1 = df[columns_f].copy()

train_df, val_df = train_test_split(
    df1, stratify=df1['class_id'], test_size=0.25#, random_state=RANDOM_SEED
)

print(train_df.shape)
print(val_df.shape)

In [ ]:
df

In [ ]:
classes = df["class"].unique()
classes

El set de entrenamiento tiene la clase y el bounding box de cada imagen.


In [ ]:
df.head()

In [ ]:
train_df['class'].value_counts(1) * 100

In [ ]:
val_df['class_id'].value_counts(1) * 100

El set de test solo tiene el nombre del archivo — hay que generar las predicciones y subirlas a Kaggle.


In [ ]:
transform_func_inp_signature = ty.Dict[str, NDArray[np.float64]]
transform_func_signature = ty.Callable[
    [transform_func_inp_signature],
    transform_func_inp_signature
]

class CarsDataset(Dataset):
    """
    Location Cars dataset
    """
    def __init__(
        self, 
        df: pd.DataFrame, 
        root_dir: str, 
        labeled: bool = True,
        transform: ty.Optional[ty.List[transform_func_signature]] = None
    ) -> None:
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
        self.labeled = labeled
        
    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, idx: int) -> transform_func_signature: 
        if torch.is_tensor(idx):
            idx = idx.tolist()
        
        # Read image
        img_name = os.path.join(self.root_dir, self.df.filename.iloc[idx])
        image = io.imread(img_name)
        
        #print(f"Dimensiones originales de la imagen: {image.shape}")  # Agregar para depuración

        if image.ndim == 2:  # Si la imagen está en escala de grises
            image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)  # Convertir a RGB
        elif image.shape[2] == 4:  # Si la imagen es RGBA
            image = image[:, :, :3] 
        
        sample = {'image': image}
        
        if self.labeled:
            # Read labels
            img_class = self.df.class_id.iloc[idx]
            #name_class = self.df['class'].iloc[idx]
            img_bbox = self.df.iloc[idx, 1:5]
            #img_h = self.df.h.iloc[idx]
            #img_w = self.df.w.iloc[idx]

            img_bbox = np.array([img_bbox]).astype('float')
            img_class = np.array([img_class]).astype('int')
            #img_h = np.array([img_h]).astype('int')
            #img_w = np.array([img_w]).astype('int')
            sample.update({'bbox': img_bbox, 'class_id': img_class})#,'name_class':name_class,'w': img_w, 'h': img_h
        
        if self.transform:
            sample = self.transform(sample)
        
        return sample

In [ ]:
def draw_bbox(img, bbox, color):
    xmin, ymin, xmax, ymax = bbox
    img = cv2.rectangle(img, (xmin, ymin), (xmax, ymax), color, 2)
    return img

def normalize_bbox(bbox, factor : int = 416):
    return list(map(lambda x: int(x * factor), bbox))
'''
def normalize_bbox(bbox, factor_h: int =1 ,factor_w: int=1):
    # Iterar a través de la lista utilizando un bucle for con el índice
    for i in range(len(bbox)):
        if i % 2 == 0:  # Comprobar si el índice es par
            nuevo_valor = int(bbox[i]*factor_w).astype('int')
        else:  # El índice es impar
            nuevo_valor = int(bbox[i]*factor_h).astype('int')
        bbox[i] = int(nuevo_valor).astype('int')
        print(f"Valor: {bbox[i]}, Tipo: {type(bbox[i])}")
    return bbox
    #return list(map(lambda x: int(x * factor), bbox))
'''

def draw_bboxes(imgs, bboxes, colors):
    for i, (img, bbox, color) in enumerate(zip(imgs, bboxes, colors)):
        imgs[i] = draw_bbox(img, bbox, color)
    return imgs

def draw_classes(imgs, classes, colors, origin, offset: int = 5, prefix: str =''):
    for i, (img, class_id, color) in enumerate(zip(imgs, classes, colors)): 
        if type(c)==list:
            name_class_=id2obj[classes[i]]
        else:
            name_class_=id2obj[classes[i][0]]
        imgs[i] = cv2.putText(
            img, f'{prefix}{name_class_}', #class_id.squeeze()
            origin, cv2.FONT_HERSHEY_SIMPLEX, 
            0.4, color, 1, cv2.LINE_AA
        )
    return imgs

def draw_predictions(imgs, classes, bboxes, colors, origin):
    assert all(len(x) > 0 for x in [imgs, classes, bboxes, colors])
    if len(colors) == 1:
        colors = [colors[0] for _ in imgs]
    imgs = draw_bboxes(imgs, bboxes, colors)
    imgs = draw_classes(imgs, classes, colors, origin)
    return imgs

In [ ]:
train_root_dir = osp.join(DATA_DIR, "cars/cars_resize")#, "train"
train_ds = CarsDataset(train_df, root_dir=train_root_dir)

num_imgs = 6
start_idx = 0

samples = [train_ds[i] for i in range(start_idx, num_imgs)]

imgs = [s['image'] for s in samples]
bboxes = [normalize_bbox(s['bbox'].squeeze()) for s in samples]
classes = [s['class_id'] for s in samples]

imgs = draw_predictions(imgs, classes, bboxes, [(0, 150, 0)], (5, 10))#(150, 10)

fig = plt.figure(figsize=(30, num_imgs))

for i, img in enumerate(imgs):
    fig.add_subplot(1, num_imgs, i+1)
    plt.imshow(img)

plt.show()

# 3. Normalización e Image Transforms

# Normalización


In [ ]:
train_ds = CarsDataset(train_df, root_dir=train_root_dir)

means = np.zeros(3)
stds = np.zeros(3)
n_images = 0

for x in train_ds:
    img = x['image']#astype(np.float32)  # Asegúrate de que la imagen está en float para cálculos precisos
    n_images += 1

    for channel in range(3):
        channel_pixels = img[..., channel]
        # Acumular la suma y suma de cuadrados para calcular la media y desviación estándar
        means[channel] += np.mean(channel_pixels)
        stds[channel] += np.std(channel_pixels)

# Calcular la media y desviación estándar final
means /= n_images
stds /= n_images

In [ ]:
print(means)
print(stds)

In [ ]:
img_filename = osp.join(DATA_DIR, "cars/cars_resize",'6_train.jpg')
img1 = cv2.imread(img_filename)
#img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)

print(img1.shape)
print(img1.transpose((2,0,1)).shape)
img1[..., 0] = 0
img1[..., 1] = 0

plt.imshow(img1)

# Transforms


## Justificación del Aumento de Datos (Data Augmentation)

Se utilizó la librería [Albumentations](https://albumentations.ai/docs/3-basic-usage/bounding-boxes-augmentations/), que transforma automáticamente las coordenadas del bounding box junto con la imagen, evitando inconsistencias entre imagen y anotación.

| Transformación | Probabilidad | Justificación |
|---|---|---|
| `HorizontalFlip` | 0.5 | Los automóviles son simétricamente plausibles en ambas orientaciones. p=0.5 genera variabilidad sin sesgar el dataset (se mantienen imágenes originales). Se corrigió de p=1.0 (bug previo que volteaba **todas** las imágenes). |
| `Rotate(±20°)` | 0.5 | Las cámaras no capturan siempre el vehículo perfectamente horizontal. Rotaciones pequeñas simulan ángulos de captura reales y mejoran la robustez ante variaciones de perspectiva. |
| `Resize(416×416)` | 1.0 | Obligatorio para uniformizar el tamaño de entrada del backbone. Albumentations ajusta las coordenadas del bbox proporcionalmente. |

**Transformaciones descartadas y razón:**
- `RandomCrop`: podría eliminar o truncar el vehículo, perdiendo el objeto de interés.
- `VerticalFlip`: no representa situaciones reales (los automóviles no aparecen invertidos).
- `ColorJitter` agresivo: el color es características distintiva por modelo de vehículo.

Las transformaciones **solo se aplican al set de entrenamiento**. El set de validación y test usan únicamente `ToTensor` + normalización, garantizando una evaluación limpia sin fuga de datos.


In [ ]:
class ToTensor(object):
    """Convert ndarrays in sample to Tensors."""

    def __call__(self, sample):
        image = sample['image']

        # swap color axis because
        # numpy image: H x W x C (0,1,2)
        # torch image: C x H x W
        image = image.transpose((2, 0, 1))
        image = torch.from_numpy(image).float()
        sample.update({'image': image})
        return sample

class Normalizer(object):
    
    def __init__(self, stds, means):
        self.stds = stds
        self.means = means
    
    def __call__(self, sample):
        image = sample['image']
        
        for channel in range(3):
            image[channel] = (image[channel] - self.means[channel]) / (self.stds[channel] + 1e-7)

        sample['image'] = image
        return sample

class TVTransformWrapper(object):
    """Torch Vision Transform Wrapper
    """
    def __init__(self, transform: torch.nn.Module):
        self.transform = transform
        
    def __call__(self, sample):
        sample['image'] = self.transform(sample['image'])
        return sample

class AlbumentationsWrapper(object):
    
    def __init__(self, transform):
        self.transform = transform
    
    def __call__(self, sample):
        # Clipear y filtrar bboxes degeneradas antes de pasar a albumentations.
        # Evita ValueError: x_max <= x_min para bbox [0,0,0,0] o fuera de [0,1].
        raw_bboxes = sample['bbox']
        valid_bboxes = []
        for bb in raw_bboxes:
            xmin = max(0.0, min(float(bb[0]), 1.0))
            ymin = max(0.0, min(float(bb[1]), 1.0))
            xmax = max(0.0, min(float(bb[2]), 1.0))
            ymax = max(0.0, min(float(bb[3]), 1.0))
            if xmax > xmin and ymax > ymin:
                valid_bboxes.append([xmin, ymin, xmax, ymax])

        if len(valid_bboxes) == 0:
            return sample

        transformed = self.transform(
            image=sample['image'],
            bboxes=valid_bboxes,
        )
        sample['image'] = transformed['image']
        if len(transformed['bboxes']) > 0:
            sample['bbox'] = np.array(transformed['bboxes'])
        return sample


In [ ]:
common_transforms = [    
    ToTensor(),
    Normalizer(
        means=means,
        stds=stds,
    )
]

# Normalizacion ImageNet para modelos preentrenados (VGG16, ResNet50, EfficientNet, ConvNeXt)
# Estos backbones fueron entrenados con esta normalizacion especifica de ImageNet.
# Usar la normalizacion incorrecta degrada las representaciones aprendidas.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
common_transforms_imagenet = [
    ToTensor(),
    Normalizer(means=IMAGENET_MEAN, stds=IMAGENET_STD)
]

# -------------------------------------------------------
# DATA AUGMENTATION
#
# HorizontalFlip (p=0.5): autos son simétricos. Bbox ajustado automáticamente.
# RandomBrightnessContrast (p=0.4): variaciones de iluminación reales (sol, interior).
# HueSaturationValue (p=0.3): variaciones de cámara. Límites pequeños.
# Affine (p=0.4): traslación y escala leve, SIN rotación (rotate=0).
#   No usar rotaciones: alteran la orientación natural del vehículo.
# Resize (p=1): normalizar a 416×416.
# min_visibility=0.3: descartar bbox con menos del 30% visible post-transformación.
#
# NO usadas: Rotate, MotionBlur — borran logos y faros clave para clasificar modelos.
# -------------------------------------------------------
train_data_augmentations = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=12, val_shift_limit=8, p=0.3),
    A.Affine(translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
             scale=(0.9, 1.1), rotate=0, mode=0, p=0.4),
    A.Resize(height=416, width=416, p=1)
    ],
    bbox_params=A.BboxParams(
        format='albumentations', 
        label_fields=[],
        min_visibility=0.3,
    )
)

# Para CNN personalizado (estadisticas del dataset)
train_transforms = torchvision.transforms.Compose(
    [AlbumentationsWrapper(train_data_augmentations)] + common_transforms
)
eval_transforms = torchvision.transforms.Compose(common_transforms)

# Para modelos preentrenados (normalizacion ImageNet)
train_transforms_imagenet = torchvision.transforms.Compose(
    [AlbumentationsWrapper(train_data_augmentations)] + common_transforms_imagenet
)
eval_transforms_imagenet = torchvision.transforms.Compose(common_transforms_imagenet)


In [ ]:
train_ds = CarsDataset(train_df, root_dir=train_root_dir)

x = next(iter(train_ds))
x_transformed = copy.deepcopy(x)
x_transformed = train_transforms(x_transformed)

original_img = x['image']
transformed_img = x_transformed['image'].numpy().transpose(1, 2, 0)

original_img = draw_bbox(
    original_img,
    normalize_bbox(x['bbox'].squeeze()),
    (0, 255, 0)
)

transformed_img = draw_bbox(
    transformed_img,
    normalize_bbox(x_transformed['bbox'].squeeze()),
    (0, 255, 0)
)

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))

axes[0].imshow(original_img)
axes[0].set_title('Original digit')

axes[1].imshow(transformed_img)
axes[1].set_title('Transformed digit')

plt.show()

# 4. Modelos de Backbone

## 1. CNN Personalizado (desde cero)

Se diseña un backbone convolucional propio **sin usar pesos preentrenados**. La arquitectura consiste en 4 bloques **Conv2d → BatchNorm2d → ReLU → MaxPool** que reducen progresivamente la resolución espacial mientras incrementan los mapas de características (32 → 64 → 128 → 256 canales). Un `AdaptiveAvgPool2d(7,7)` al final fija el tamaño de salida independientemente del input. BatchNorm estabiliza el entrenamiento desde cero y actúa como regularizador.


In [ ]:
# =========================
# 1. CNN Personalizado (desde cero)
# Sin pesos preentrenados — entrenado completamente desde cero
# =========================

class CustomCNNBackbone(nn.Module):
    """
    CNN personalizado desde cero.
    5 bloques Conv2d-BatchNorm2d-ReLU con MaxPool.
    Global Average Pool (1×1) al final → vector compacto sin pérdida de info espacial.
    """
    def __init__(self, dropout_p: float = 0.3):
        super().__init__()
        self.features = nn.Sequential(
            # Bloque 1: 3→32ch, 416×416 → 208×208
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            # Bloque 2: 32→64ch, 208×208 → 104×104
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            # Bloque 3: 64→128ch, 104×104 → 52×52
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            # Bloque 4: 128→256ch, 52×52 → 26×26
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            # Bloque 5: 256→512ch, 26×26 → 13×13
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            # Global Average Pool → (B, 512, 1, 1)
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=dropout_p)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.flatten(x)
        x = self.dropout(x)
        return x

custom_cnn_backbone = CustomCNNBackbone(dropout_p=0.3).to(device)
summary(custom_cnn_backbone, (3, 416, 416))


## 2. Transfer Learning – VGG16

Se usa VGG16 preentrenado en ImageNet. Los primeros 4 bloques convolucionales se congelan y solo el último bloque (`features[24:]`) se desbloquea para fine-tuning, permitiendo que el modelo se adapte al dominio de automóviles Chevrolet con un lr conservador (`1e-4`).


In [ ]:
# =========================
# VGG16 Feature Extractor
# - Freeze first 4 blocks (features[:24]), unfreeze last block (features[24:])
# - Reduced dropout to 0.3 to avoid underfitting
# =========================

import torch
from torch import nn
from torchvision.models import vgg16, VGG16_Weights

# ---- dispositivo (asegúrate de tener 'device' definido)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class FeatureExtractor(nn.Module):
    def __init__(self, backbone: nn.Module, freeze_backbone: bool = True, dropout_p: float = 0.3):
        super().__init__()

        # VGG16 conv blocks
        self.features = backbone.features  # nn.Sequential
        # Average pooling layer
        self.pooling = backbone.avgpool
        # Flatten + Dropout
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=dropout_p)

        if freeze_backbone:
            # Freeze only the first 4 blocks (features[:24]), keep block 5 (features[24:]) trainable
            for p in self.features[:24].parameters():
                p.requires_grad = False
            for p in self.features[24:].parameters():
                p.requires_grad = True
            for p in self.pooling.parameters():
                p.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.pooling(x)
        x = self.flatten(x)
        x = self.dropout(x)
        return x

# ---- cargar pesos modernos (elimina warnings)
weights = VGG16_Weights.DEFAULT
vgg16_model = vgg16(weights=weights)

# ---- crear extractor: block 5 descongelado, dropout reducido
pretrained_model = FeatureExtractor(vgg16_model, freeze_backbone=True, dropout_p=0.3).to(device)

print(pretrained_model)


## 3. Transfer Learning – ResNet50

Se usa ResNet50 como segundo backbone preentrenado. Sus **conexiones residuales** permiten entrenar redes más profundas sin degradación del gradiente (vanishing gradient). Se congela todo excepto `layer4` (último bloque residual) para el fine-tuning, permitiendo adaptar las características de alto nivel al dominio de automóviles.


In [ ]:
# =========================
# 3. Transfer Learning – ResNet50
# Segundo modelo preentrenado para comparación con VGG16
# Se reemplaza backbone.fc por nn.Identity() para eliminar el clasificador
# de forma segura (metodo recomendado por torchvision).
# Fine-tuning: layer4 descongelada para adaptar al dominio vehiculos.
# =========================
from torchvision.models import resnet50, ResNet50_Weights

class ResNet50FeatureExtractor(nn.Module):
    """
    ResNet50 como backbone preentrenado. Salida: 2048 features.
    Se reemplaza backbone.fc por nn.Identity() para eliminar el clasificador
    de forma segura (metodo recomendado por torchvision).
    Fine-tuning: layer4 descongelada para adaptar al dominio vehiculos.
    """
    def __init__(self, freeze_backbone: bool = True, dropout_p: float = 0.3):
        super().__init__()
        backbone    = resnet50(weights=ResNet50_Weights.DEFAULT)
        backbone.fc = nn.Identity()   # eliminar clasificador final de forma segura
        self.backbone = backbone
        self.dropout  = nn.Dropout(p=dropout_p)

        if freeze_backbone:
            # Congelar todo excepto layer4 (ultimas capas residuales)
            for name, p in self.backbone.named_parameters():
                if 'layer4' not in name:
                    p.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.backbone(x)   # shape: (B, 2048)
        x = self.dropout(x)
        return x

resnet50_backbone = ResNet50FeatureExtractor(freeze_backbone=True, dropout_p=0.3).to(device)
print("ResNet50 Feature Extractor (layer4 descongelada para fine-tuning):")
summary(resnet50_backbone, (3, 416, 416))


---
## Backbones 3 y 4: EfficientNet-B3 y ConvNeXt-Tiny (preentrenados en ImageNet)

**EfficientNet-B3:** escala profundidad/ancho/resolución de forma compuesta.
- ~82% top-1 ImageNet, solo ~12M parámetros. Fine-tuning de los últimos 2 bloques MBConv.

**ConvNeXt-Tiny:** arquitectura convolucional moderna inspirada en ViT.
- 82.1% top-1 ImageNet, patrón seguro: `backbone.classifier = nn.Identity()`. Fine-tuning de las últimas 2 etapas.

Ambos usan normalización ImageNet y learning rate diferenciado (backbone muy bajo para no olvidar características aprendidas).

In [ ]:
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

class EfficientNetB3FeatureExtractor(nn.Module):
    """EfficientNet-B3 preentrenado. Salida: 1536 features."""
    def __init__(self, freeze_backbone=True, unfreeze_last_n=2, dropout_p=0.3):
        super().__init__()
        backbone      = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
        self.features = backbone.features
        self.avgpool  = backbone.avgpool
        self.flatten  = nn.Flatten()
        self.dropout  = nn.Dropout(p=dropout_p)
        for p in self.features.parameters():
            p.requires_grad = False
        if unfreeze_last_n > 0:
            n_total = len(list(self.features.children()))
            for blk in list(self.features.children())[n_total - unfreeze_last_n:]:
                for p in blk.parameters():
                    p.requires_grad = True

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        return x


class ConvNeXtTinyFeatureExtractor(nn.Module):
    """
    ConvNeXt-Tiny preentrenado. Salida: 768 features.
    82.1% top-1 ImageNet. Patron seguro: backbone.classifier = Identity().
    backbone(x) devuelve (B, 768) directamente (avgpool+LN+flatten internos).
    Fine-tuning de las ultimas 2 etapas del backbone.
    """
    def __init__(self, freeze_backbone=True, unfreeze_last_n_stages=2, dropout_p=0.3):
        super().__init__()
        backbone = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        self.dropout  = nn.Dropout(p=dropout_p)
        for p in self.backbone.parameters():
            p.requires_grad = False
        if unfreeze_last_n_stages > 0:
            feat_children = list(self.backbone.features.children())
            n = len(feat_children)
            for blk in feat_children[n - unfreeze_last_n_stages * 2:]:
                for p in blk.parameters():
                    p.requires_grad = True

    def forward(self, x):
        x = self.backbone(x)
        x = self.dropout(x)
        return x


# ---- EfficientNet-B3 ----
efficientnet_backbone = EfficientNetB3FeatureExtractor(
    freeze_backbone=True, unfreeze_last_n=2, dropout_p=0.3
).to(device)
print("EfficientNet-B3:")
summary(efficientnet_backbone, (3, 416, 416))

# ---- ConvNeXt-Tiny ----
convnext_backbone = ConvNeXtTinyFeatureExtractor(
    freeze_backbone=True, unfreeze_last_n_stages=2, dropout_p=0.3
).to(device)
print("\nConvNeXt-Tiny:")
summary(convnext_backbone, (3, 416, 416))


# 5. Modelo Multi-tarea
`get_output_shape` + clase `Model` (cabezas de clasificación y regresión).

In [ ]:
def get_output_shape(model: nn.Sequential, image_dim: ty.Tuple[int, int, int]):
    return model(torch.rand(*(image_dim)).to(device)).data.shape

class Model(nn.Module):
    def __init__(
        self,
        backbone: nn.Module,
        input_shape: ty.Tuple[int, int, int] = (3, 416, 416),
        n_classes: int = 22,
        hidden_dim: int = 512,   # neuronas en capas ocultas
        dropout_p: float = 0.3,  # dropout reducido vs 0.5 original
    ):
        """
        Modelo multitarea con dos salidas compartiendo el mismo backbone:
            1. Clasificación del tipo de vehículo (cls_head).
            2. Regresión del bounding box (reg_head).

        Acepta cualquier backbone como extractor de características,
        lo que permite comparar CNN personalizado, VGG16 y ResNet50
        manteniendo las mismas cabezas.
        """
        super().__init__()
        self.input_shape = input_shape
        self.backbone = backbone

        backbone_output_shape = get_output_shape(self.backbone, [1, *input_shape])
        backbone_output_features = reduce(lambda x, y: x*y, backbone_output_shape)

        # BatchNorm1d estabiliza el entrenamiento en las cabezas densas
        self.cls_head = nn.Sequential(
            nn.Linear(backbone_output_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim // 2, n_classes)
        )

        # Sigmoid al final garantiza predicciones de bbox en [0,1],
        # coherente con coordenadas normalizadas del dataset
        self.reg_head = nn.Sequential(
            nn.Linear(backbone_output_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim // 2, 4),
            nn.Sigmoid()
        )

    def forward(self, x: Tensor) -> ty.Dict[str, Tensor]:
        features = self.backbone(x)
        features = features.view(features.size(0), -1)
        cls_logits = self.cls_head(features)
        pred_bbox = self.reg_head(features)
        return {'bbox': pred_bbox, 'class_id': cls_logits}


In [ ]:
# Test rápido con el backbone CNN personalizado
# x del dataset crudo es (H, W, C) numpy; eval_transforms convierte a tensor (C, H, W)
import copy as _copy
x_t = eval_transforms(_copy.deepcopy(x))
img_tensor = x_t['image'].unsqueeze(0).to(device)  # (1, C, H, W)
print('image shape:', img_tensor.shape)
test_model = Model(backbone=custom_cnn_backbone, input_shape=(3, 416, 416), n_classes=22).to(device)
preds = test_model(img_tensor)
print(preds)


# 6. Infraestructura de Entrenamiento

# Métricas


In [ ]:
def iou(y_true: Tensor, y_pred: Tensor):
    pairwise_iou = torchvision.ops.box_iou(y_true.squeeze(), y_pred.squeeze())
    result = torch.trace(pairwise_iou) / pairwise_iou.size()[0]
    return result

In [ ]:
def accuracy(y_true: Tensor, y_pred: Tensor):
    pred = torch.argmax(y_pred, axis=-1)
    y_true = y_true.squeeze()
    correct = torch.eq(pred, y_true).float()
    total = torch.ones_like(correct)
    result = torch.divide(torch.sum(correct), torch.sum(total))
    return result

# Loss


In [ ]:
def loss_fn(y_true, y_preds, alpha: float = 0.6):
    # Se elimina .unsqueeze(-1) — causaba shape (B,n_classes,1) en lugar de (B,n_classes)
    # F.cross_entropy espera (B, n_classes), el bug impedía que la clasificación aprendiera.
    cls_y_true = y_true['class_id'].long().squeeze()   # shape: (B,)
    cls_y_pred = y_preds['class_id'].float()           # shape: (B, n_classes)

    reg_y_true, reg_y_pred = y_true['bbox'].float().squeeze(), y_preds['bbox'].float().squeeze()
    
    cls_loss = F.cross_entropy(cls_y_pred, cls_y_true, label_smoothing=0.1)
    
    # Smooth L1 is more robust to outliers than MSE for bbox regression
    reg_loss = F.smooth_l1_loss(reg_y_pred, reg_y_true)
    # alpha=0.6: 60% peso en regresión bbox, 40% en clasificación
    total_loss = (1 - alpha) * cls_loss + alpha * reg_loss
    return dict(loss=total_loss, reg_loss=reg_loss, cls_loss=cls_loss)


# Callbacks


In [ ]:
def printer(logs: ty.Dict[str, ty.Any]):
    # print every 10 steps
    if logs['iters'] % 10 != 0:
        return
    print('Iteration #: ',logs['iters'])
    for name, value in logs.items():
        if name == 'iters':
            continue
        
        if type(value) in [float, int]:
            value = round(value, 4)
        elif type(value) is torch.Tensor:
            value = torch.round(value, decimals=4)
        
        print(f'\t{name} = {value}')
    print()

# Loop de entrenamiento


In [ ]:
def evaluate(
    logs: ty.Dict[str, ty.Any], 
    labels: ty.Dict[str, Tensor],
    preds: ty.Dict[str, Tensor],
    eval_set: str,
    metrics: ty.Dict[str, ty.Callable[[Tensor, Tensor], Tensor]],
    losses: ty.Optional[ty.Dict[str, Tensor]] = None,
) -> ty.Dict[str, ty.Any]:
    
    if losses is not None:
        for loss_name, loss_value in losses.items():
            logs[f'{eval_set}_{loss_name}'] = loss_value
    
    for task_name, label in labels.items():
        for metric_name, metric in metrics[task_name]:
            value = metric(label, preds[task_name])
            logs[f'{eval_set}_{metric_name}'] = value
            
    return logs

def step(
    model: Model, 
    optimizer: Optimizer, 
    batch: CarsDataset,
    loss_fn: ty.Callable[[ty.Dict[str, torch.Tensor]], torch.Tensor],
    device: str,
    train: bool = False,
    alpha: float = 0.6,
) -> ty.Tuple[ty.Dict[str, Tensor], ty.Dict[str, Tensor]]:
    
    if train:
        optimizer.zero_grad()
    
    img = batch.pop('image').to(device)
    
    for k in list(batch.keys()):
        batch[k] = batch[k].to(device)
    
    preds  = model(img.float())
    losses = loss_fn(batch, preds, alpha=alpha)

    if train:
        losses['loss'].backward()
        # Gradient clipping: previene explosión de gradientes al inicio del entrenamiento
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
    
    return losses, preds


def train(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    dataset: DataLoader,
    eval_datasets: ty.List[ty.Tuple[str, DataLoader]],
    loss_fn: ty.Callable[[ty.Dict[str, torch.Tensor]], torch.Tensor],
    metrics: ty.Dict[str, ty.Callable[[torch.Tensor, torch.Tensor], torch.Tensor]],
    callbacks: ty.List[ty.Callable[[ty.Dict[ty.Any, ty.Any]], None]],
    device: str,
    train_steps: int = 100,
    eval_steps: int = 10,
    save_path: str = "best_model.pth",
    alpha: float = 0.6,
    scheduler: ty.Optional[object] = None  # CosineAnnealingLR opcional
) -> nn.Module:
    model = model.to(device)
    iters    = 0
    iterator = iter(dataset)
    assert train_steps > eval_steps, "train_steps debe ser mayor que eval_steps"

    best_loss = float("inf")

    while iters <= train_steps:
        logs = dict()
        logs["iters"] = iters

        try:
            batch = next(iterator)
        except StopIteration:
            iterator = iter(dataset)
            batch    = next(iterator)

        model.train()
        losses, preds = step(model, optimizer, batch, loss_fn, device,
                             train=True, alpha=alpha)
        logs = evaluate(logs, batch, preds, "train", metrics, losses)

        if scheduler is not None:
            scheduler.step()

        current_loss = losses["loss"]
        if current_loss < best_loss:
            best_loss = current_loss
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': best_loss
            }, save_path)
            print(f"New best loss: {best_loss:.4f}. Model saved to {save_path}.")

        if iters % eval_steps == 0:
            model.eval()
            with torch.no_grad():
                for name, eval_ds in eval_datasets:
                    for batch in eval_ds:
                        losses_v, preds_v = step(model, optimizer, batch, loss_fn,
                                                  device, train=False, alpha=alpha)
                        logs = evaluate(logs, batch, preds_v, name, metrics, losses_v)

        for callback in callbacks:
            callback(logs)

        iters += 1

    return model


# 7. Entrenamiento

# Ejecución


In [ ]:
# Hparams base
batch_size = 32

# DataLoaders para CNN personalizado (normalizacion estadisticas del dataset)
train_ds   = CarsDataset(train_df, root_dir=train_root_dir, transform=train_transforms)
val_ds     = CarsDataset(val_df,   root_dir=train_root_dir, transform=eval_transforms)
train_data = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=cpu_count())
val_data   = DataLoader(val_ds,   batch_size=batch_size, num_workers=cpu_count())

# DataLoaders para modelos preentrenados (normalizacion ImageNet)
train_ds_tl   = CarsDataset(train_df, root_dir=train_root_dir, transform=train_transforms_imagenet)
val_ds_tl     = CarsDataset(val_df,   root_dir=train_root_dir, transform=eval_transforms_imagenet)
train_data_tl = DataLoader(train_ds_tl, batch_size=batch_size, shuffle=True,  num_workers=cpu_count())
val_data_tl   = DataLoader(val_ds_tl,   batch_size=batch_size, num_workers=cpu_count())


> ## ⚠️ NO EJECUTAR — Entrenamiento comparativo (~2 horas)
>
> Esta celda entrena los **5 modelos desde cero**.
>
> **Si ya tienes los `.pth` en `/kaggle/working/`, SALTA esta celda** y ejecuta directamente la celda de **Carga de Checkpoints** (más abajo).
>
> Archivos necesarios:
> - `best_model_custom_cnn.pth`
> - `best_model_vgg16.pth`
> - `best_model_resnet50.pth`
> - `best_model_efficientnet_b3.pth`
> - `best_model_convnext_tiny.pth`


In [ ]:
# =========================
# Entrenamiento de los 5 backbones con sus hiperparametros optimos
# =========================

# --- 1. CNN Personalizado ---
print("=" * 70)
print("Experimento 1: CNN Personalizado | normalizacion=dataset | alpha=0.6")
print("=" * 70)
torch.cuda.empty_cache()
cnn_backbone_exp = CustomCNNBackbone(dropout_p=0.3).to(device)
model_custom = Model(
    input_shape=(3, 416, 416), n_classes=22,
    backbone=cnn_backbone_exp, hidden_dim=512, dropout_p=0.3
).to(device)
optimizer_custom       = torch.optim.Adam(lr=1e-3, params=model_custom.parameters())
scheduler_custom       = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_custom, T_max=20 * 100, eta_min=1e-6
)
name_save_model_custom = "best_model_custom_cnn.pth"

for epoch in range(20):
    print(f'Epoch {epoch + 1}/20')
    model_custom = train(
        model_custom, optimizer_custom, train_data,
        eval_datasets=[('val', val_data)], loss_fn=loss_fn,
        metrics={'bbox': [('iou', iou)], 'class_id': [('accuracy', accuracy)]},
        callbacks=[printer], device=device,
        train_steps=100, eval_steps=10,
        save_path=name_save_model_custom, alpha=0.6,
        scheduler=scheduler_custom
    )
print("Experimento 1 (CNN) finalizado.")

# --- 2. Transfer Learning VGG16 ---
print("=" * 70)
print("Experimento 2: VGG16 TL | normalizacion=ImageNet | alpha=0.6 | 30 epocas")
print("=" * 70)
torch.cuda.empty_cache()
vgg_backbone_exp = FeatureExtractor(
    vgg16(weights=VGG16_Weights.DEFAULT), freeze_backbone=True, dropout_p=0.3
).to(device)
model_vgg = Model(
    input_shape=(3, 416, 416), n_classes=22,
    backbone=vgg_backbone_exp, hidden_dim=512, dropout_p=0.3
).to(device)
optimizer_vgg       = torch.optim.Adam(lr=1e-4, params=model_vgg.parameters())
scheduler_vgg       = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_vgg, T_max=30 * 100, eta_min=1e-6
)
name_save_model_vgg = "best_model_vgg16.pth"

for epoch in range(30):
    print(f'Epoch {epoch + 1}/30')
    model_vgg = train(
        model_vgg, optimizer_vgg, train_data_tl,
        eval_datasets=[('val', val_data_tl)], loss_fn=loss_fn,
        metrics={'bbox': [('iou', iou)], 'class_id': [('accuracy', accuracy)]},
        callbacks=[printer], device=device,
        train_steps=100, eval_steps=10,
        save_path=name_save_model_vgg, alpha=0.6,
        scheduler=scheduler_vgg
    )
print("Experimento 2 (VGG16) finalizado.")

# --- 3. Transfer Learning ResNet50 ---
print("=" * 70)
print("Experimento 3: ResNet50 TL+FT | normalizacion=ImageNet | alpha=0.35 | 40 epocas")
print("  alpha=0.35: mayor peso en clasificacion")
print("=" * 70)
torch.cuda.empty_cache()
rn50_backbone_exp = ResNet50FeatureExtractor(freeze_backbone=True, dropout_p=0.3).to(device)
model_resnet50 = Model(
    input_shape=(3, 416, 416), n_classes=22,
    backbone=rn50_backbone_exp, hidden_dim=512, dropout_p=0.3
).to(device)
optimizer_resnet50       = torch.optim.Adam(lr=1e-4, params=model_resnet50.parameters())
scheduler_resnet         = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_resnet50, T_max=40 * 100, eta_min=1e-6
)
name_save_model_resnet50 = "best_model_resnet50.pth"

for epoch in range(40):
    print(f'Epoch {epoch + 1}/40')
    model_resnet50 = train(
        model_resnet50, optimizer_resnet50, train_data_tl,
        eval_datasets=[('val', val_data_tl)], loss_fn=loss_fn,
        metrics={'bbox': [('iou', iou)], 'class_id': [('accuracy', accuracy)]},
        callbacks=[printer], device=device,
        train_steps=100, eval_steps=10,
        save_path=name_save_model_resnet50, alpha=0.35,
        scheduler=scheduler_resnet
    )
print("Experimento 3 (ResNet50) finalizado.")

# --- 4. EfficientNet-B3 ---
print("=" * 70)
print("Experimento 4: EfficientNet-B3 | normalizacion=ImageNet | alpha=0.40 | 30 epocas")
print("=" * 70)
torch.cuda.empty_cache()
effnet_backbone_exp = EfficientNetB3FeatureExtractor(
    freeze_backbone=True, unfreeze_last_n=2, dropout_p=0.3
).to(device)
model_efficientnet = Model(
    input_shape=(3, 416, 416), n_classes=22,
    backbone=effnet_backbone_exp, hidden_dim=512, dropout_p=0.3
).to(device)
bk_eff = [p for p in model_efficientnet.backbone.parameters() if p.requires_grad]
hd_eff = list(model_efficientnet.cls_head.parameters()) + list(model_efficientnet.reg_head.parameters())
optimizer_efficientnet       = torch.optim.Adam([{'params': bk_eff, 'lr': 2e-5},
                                                  {'params': hd_eff, 'lr': 1e-4}])
scheduler_eff                = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_efficientnet, T_max=30 * 100, eta_min=1e-6
)
name_save_model_efficientnet = "best_model_efficientnet_b3.pth"

for epoch in range(30):
    print(f'Epoch {epoch + 1}/30')
    model_efficientnet = train(
        model_efficientnet, optimizer_efficientnet, train_data_tl,
        eval_datasets=[('val', val_data_tl)], loss_fn=loss_fn,
        metrics={'bbox': [('iou', iou)], 'class_id': [('accuracy', accuracy)]},
        callbacks=[printer], device=device,
        train_steps=100, eval_steps=10,
        save_path=name_save_model_efficientnet, alpha=0.40,
        scheduler=scheduler_eff
    )
print("Experimento 4 (EfficientNet-B3) finalizado.")

# --- 5. ConvNeXt-Tiny ---
print("=" * 70)
print("Experimento 5: ConvNeXt-Tiny | normalizacion=ImageNet | alpha=0.30 | 50 epocas")
print("  alpha=0.30: potenciar clasificacion (Acc >= 0.88)")
print("=" * 70)
torch.cuda.empty_cache()
cnx_backbone_exp = ConvNeXtTinyFeatureExtractor(
    freeze_backbone=True, unfreeze_last_n_stages=2, dropout_p=0.3
).to(device)
model_convnext = Model(
    input_shape=(3, 416, 416), n_classes=22,
    backbone=cnx_backbone_exp, hidden_dim=512, dropout_p=0.3
).to(device)
bk_cnx = [p for p in model_convnext.backbone.parameters() if p.requires_grad]
hd_cnx = list(model_convnext.cls_head.parameters()) + list(model_convnext.reg_head.parameters())
optimizer_convnext       = torch.optim.Adam([{'params': bk_cnx, 'lr': 2e-5},
                                              {'params': hd_cnx, 'lr': 1e-4}])
scheduler_cnx            = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_convnext, T_max=50 * 100, eta_min=1e-6
)
name_save_model_convnext = "best_model_convnext_tiny.pth"

for epoch in range(50):
    print(f'Epoch {epoch + 1}/50')
    model_convnext = train(
        model_convnext, optimizer_convnext, train_data_tl,
        eval_datasets=[('val', val_data_tl)], loss_fn=loss_fn,
        metrics={'bbox': [('iou', iou)], 'class_id': [('accuracy', accuracy)]},
        callbacks=[printer], device=device,
        train_steps=100, eval_steps=10,
        save_path=name_save_model_convnext, alpha=0.30,
        scheduler=scheduler_cnx
    )
print("Experimento 5 (ConvNeXt-Tiny) finalizado.")


## Comparación y Selección del Mejor Modelo

Se evalúan los 3 backbones con idénticos hiperparámetros (`lr=1e-4`, `batch_size=16`, `train_steps=1000`, Smooth L1 loss, ReduceLROnPlateau). La métrica principal de selección es **accuracy** (clasificación), con IoU como criterio secundario (regresión de bbox).

| Backbone | Estrategia | Características |
|----------|-----------|-----------------|
| CNN Personalizado | Desde cero | 4 bloques Conv-BN-ReLU, sin pesos previos |
| VGG16 | Transfer learning | Block 5 descongelado, 138M params totales |
| ResNet50 | Transfer learning | layer4 descongelado, conexiones residuales |

Tras la comparación, el mejor modelo se continúa entrenando por `train_steps_final=3000` pasos adicionales con `lr/2` para convergencia fina.


In [ ]:
# =========================
# Evaluacion comparativa de los 5 modelos sobre val set
# =========================
def evaluate_model_on_val(model_eval, val_loader, device, alpha=0.6):
    """Evalua un modelo sobre validacion y retorna metricas promedio."""
    model_eval.eval()
    total_iou, total_acc, total_cls, total_reg, n = 0.0, 0.0, 0.0, 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            imgs = batch.pop('image').to(device)
            for k in list(batch.keys()):
                batch[k] = batch[k].to(device)
            preds  = model_eval(imgs.float())
            losses = loss_fn(batch, preds, alpha=alpha)
            total_iou += iou(batch['bbox'],          preds['bbox']).item()
            total_acc += accuracy(batch['class_id'], preds['class_id']).item()
            total_cls += losses['cls_loss'].item()
            total_reg += losses['reg_loss'].item()
            n += 1
    return {
        'val_iou':      total_iou / n,
        'val_accuracy': total_acc / n,
        'val_cls_loss': total_cls / n,
        'val_reg_loss': total_reg / n,
    }

# Loaders especificos para evaluacion (sin augmentation)
val_loader_dataset  = DataLoader(
    CarsDataset(val_df, root_dir=train_root_dir, transform=eval_transforms),
    batch_size=32, num_workers=cpu_count()
)
val_loader_imagenet = DataLoader(
    CarsDataset(val_df, root_dir=train_root_dir, transform=eval_transforms_imagenet),
    batch_size=32, num_workers=cpu_count()
)

# Modelos con su alpha y loader correspondiente
tl_model_names = {'VGG16', 'ResNet50', 'EfficientNet-B3', 'ConvNeXt-Tiny'}
results = {}

print("\n" + "=" * 70)
print("RESULTADOS COMPARATIVOS (val set)")
print("=" * 70)

for name, mdl, al in [
    ('CNN',             model_custom,      0.6),
    ('VGG16',           model_vgg,         0.6),
    ('ResNet50',        model_resnet50,     0.35),
    ('EfficientNet-B3', model_efficientnet, 0.40),
    ('ConvNeXt-Tiny',   model_convnext,     0.30),
]:
    val_loader = val_loader_dataset if name == 'CNN' else val_loader_imagenet
    r = evaluate_model_on_val(mdl, val_loader, device, alpha=al)
    results[name] = r
    print(f"  {name:20s} → accuracy: {r['val_accuracy']:.4f} | IoU: {r['val_iou']:.4f}")

models_dict = {
    'CNN':             model_custom,
    'VGG16':           model_vgg,
    'ResNet50':        model_resnet50,
    'EfficientNet-B3': model_efficientnet,
    'ConvNeXt-Tiny':   model_convnext,
}

# Seleccion del mejor modelo individual
best_model_name = max(results, key=lambda k: results[k]['val_iou'] + results[k]['val_accuracy'])
best_iou_name   = max(results, key=lambda k: results[k]['val_iou'])
best_acc_name   = max(results, key=lambda k: results[k]['val_accuracy'])

print(f"\nMejor modelo combinado (IoU+Acc): {best_model_name}")
print(f"Mejor IoU:      {best_iou_name}  -> {results[best_iou_name]['val_iou']:.4f}")
print(f"Mejor Accuracy: {best_acc_name}  -> {results[best_acc_name]['val_accuracy']:.4f}")

best_model = models_dict[best_model_name]
bbox_model  = models_dict[best_iou_name]
class_model = models_dict[best_acc_name]

# Transforms de evaluacion correctos por modelo
bbox_eval_transforms  = eval_transforms_imagenet if best_iou_name in tl_model_names else eval_transforms
class_eval_transforms = eval_transforms_imagenet if best_acc_name in tl_model_names else eval_transforms

model = best_model  # alias para celdas de inferencia y submission
print(f"\nModelo final ({best_model_name}) listo para inferencia y submission.")
print(f"Ensemble: bbox={best_iou_name} | clase={best_acc_name}")


---
## ✅ EJECUTAR AQUÍ si saltaste el entrenamiento
### Carga de Checkpoints (sin reentrenar)

Restaura los 5 modelos desde los `.pth` guardados en `/kaggle/working/`.

Los aliases `bbox_model` (VGG16) y `class_model` (ConvNeXt) quedan listos para Submission.


In [ ]:
import os

def load_model_from_checkpoint(path, backbone, device, hidden_dim=512, dropout_p=0.3, n_classes=22):
    """Instancia un Model con el backbone dado y carga los pesos del checkpoint."""
    mdl = Model(
        input_shape=(3, 416, 416), n_classes=n_classes,
        backbone=backbone, hidden_dim=hidden_dim, dropout_p=dropout_p
    ).to(device)
    if os.path.exists(path):
        ckpt = torch.load(path, map_location=device)
        mdl.load_state_dict(ckpt['model_state_dict'])
        print(f"  ✓ {path}  (loss={ckpt.get('loss', float('nan')):.4f})")
    else:
        print(f"  ✗ {path}  NO ENCONTRADO")
    mdl.eval()
    return mdl

print("Cargando checkpoints...")

# CNN personalizado
_cnn_bb = CustomCNNBackbone(dropout_p=0.3).to(device)
model_custom = load_model_from_checkpoint("best_model_custom_cnn.pth", _cnn_bb, device)

# VGG16
_vgg_bb = FeatureExtractor(vgg16(weights=VGG16_Weights.DEFAULT), freeze_backbone=True, dropout_p=0.3).to(device)
model_vgg = load_model_from_checkpoint("best_model_vgg16.pth", _vgg_bb, device)

# ResNet50
_rn50_bb = ResNet50FeatureExtractor(freeze_backbone=True, dropout_p=0.3).to(device)
model_resnet50 = load_model_from_checkpoint("best_model_resnet50.pth", _rn50_bb, device)

# EfficientNet-B3
_eff_bb = EfficientNetB3FeatureExtractor(freeze_backbone=True, unfreeze_last_n=2, dropout_p=0.3).to(device)
model_efficientnet = load_model_from_checkpoint("best_model_efficientnet_b3.pth", _eff_bb, device)

# ConvNeXt-Tiny
_cnx_bb = ConvNeXtTinyFeatureExtractor(freeze_backbone=True, unfreeze_last_n_stages=2, dropout_p=0.3).to(device)
model_convnext = load_model_from_checkpoint("best_model_convnext_tiny.pth", _cnx_bb, device)

# Aliases para el resto del notebook
tl_model_names = {'VGG16', 'ResNet50', 'EfficientNet-B3', 'ConvNeXt-Tiny'}
models_dict = {
    'CNN':             model_custom,
    'VGG16':           model_vgg,
    'ResNet50':        model_resnet50,
    'EfficientNet-B3': model_efficientnet,
    'ConvNeXt-Tiny':   model_convnext,
}
best_model   = model_convnext
bbox_model   = model_vgg          # mejor IoU
class_model  = model_convnext     # mejor Accuracy
bbox_eval_transforms  = eval_transforms_imagenet
class_eval_transforms = eval_transforms_imagenet
model = best_model
print("\nTodos los modelos listos.")


---
# Fine-tuning adicional: ConvNeXt-Tiny

Continuar entrenando el mejor modelo desde el checkpoint guardado.
- `alpha=0.15`: 85% peso en clasificación (la métrica de Kaggle)
- `lr` backbone muy bajo (5e-6) para no destruir representaciones ImageNet
- `lr` cabezas más alto (2e-5) para adaptarlas
- 20 épocas × 100 steps = 2000 pasos adicionales

> ## ⚙️ OPCIONAL — Fine-tuning ConvNeXt-Tiny (~20 min adicionales)
>
> Mejora el score entrenando más el mejor modelo.
> Genera `best_model_convnext_ft.pth`.
>
> **Si NO ejecutas esta celda:** `class_model` ya apunta a `best_model_convnext_tiny.pth` cargado arriba — puedes ir directo a Submission.
>
> **Si SÍ ejecutas esta celda:** al terminar, `class_model` se actualiza al modelo fine-tuned automáticamente.


In [ ]:
import os, shutil
from datetime import datetime

# --- Continuar desde el mejor checkpoint disponible ---
FT_PATH   = 'best_model_convnext_ft.pth'
BASE_PATH = 'best_model_convnext_tiny.pth'

if os.path.exists(FT_PATH):
    # Cargar el ft anterior como punto de partida y hacer backup con timestamp
    ckpt = torch.load(FT_PATH, map_location=device)
    model_convnext.load_state_dict(ckpt['model_state_dict'])
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    backup_path = f'best_model_convnext_ft_{ts}.pth'
    shutil.copy(FT_PATH, backup_path)
    print(f'Continuando desde ft anterior (loss={ckpt.get("loss", float("nan")):.4f})')
    print(f'Backup guardado: {backup_path}')
else:
    print(f'No hay ft previo → partiendo desde {BASE_PATH}')

torch.cuda.empty_cache()

# Differential LR: backbone casi congelado, cabezas adaptan mas rapido
bk_cnx_ft = [p for p in model_convnext.backbone.parameters() if p.requires_grad]
hd_cnx_ft  = list(model_convnext.cls_head.parameters()) + list(model_convnext.reg_head.parameters())

optimizer_cnx_ft = torch.optim.Adam([
    {'params': bk_cnx_ft, 'lr': 5e-6},   # backbone: casi congelado
    {'params': hd_cnx_ft, 'lr': 2e-5},   # cabezas: mas rapidas
])
scheduler_cnx_ft = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_cnx_ft, T_max=20 * 100, eta_min=1e-7
)

print('=' * 70)
print('Fine-tuning ConvNeXt-Tiny | alpha=0.05 | 20 epocas x 100 steps')
print('  alpha=0.05 → 95% peso en clasificacion (metrica Kaggle)')
print('=' * 70)

for epoch in range(20):
    print(f'Epoch {epoch + 1}/20')
    model_convnext = train(
        model_convnext, optimizer_cnx_ft, train_data_tl,
        eval_datasets=[('val', val_data_tl)], loss_fn=loss_fn,
        metrics={'bbox': [('iou', iou)], 'class_id': [('accuracy', accuracy)]},
        callbacks=[printer], device=device,
        train_steps=100, eval_steps=10,
        save_path=FT_PATH, alpha=0.05,
        scheduler=scheduler_cnx_ft
    )

model = model_convnext
best_model = model_convnext
class_model = model_convnext
print(f'Fine-tuning finalizado. Checkpoint activo: {FT_PATH}')
if os.path.exists(FT_PATH):
    print(f'Backup del run anterior disponible en: {backup_path}')


# 8. Visualización de Predicciones (Opcional)
No es necesario ejecutar para la submission.

In [ ]:
num_imgs = 5
ncols = 5
nrows = math.ceil(num_imgs / ncols)

start_idx = 0

inference_ds = CarsDataset(val_df.iloc[start_idx:start_idx+num_imgs], root_dir=train_root_dir)
inference_data = DataLoader(inference_ds, batch_size=num_imgs, num_workers=1, shuffle=False)
inference_batch = next(iter(inference_data))
inference_imgs = np.empty((num_imgs, 3, 416, 416))

transform = eval_transforms

for i, img in enumerate(inference_batch['image']):
    inference_imgs[i] = transform(dict(image=img.numpy()))['image'].numpy()

preds = model(torch.tensor(inference_imgs).float().to(device))

samples = [inference_ds[i] for i in range(start_idx, num_imgs)]

imgs = [s['image'] for s in samples]
bboxes = [normalize_bbox(s['bbox'].squeeze()) for s in samples]
classes = [s['class_id'] for s in samples]

pred_bboxes = preds['bbox'].detach().cpu().numpy()
pred_bboxes = [normalize_bbox(bbox) for bbox in pred_bboxes]
pred_classes = preds['class_id'].argmax(-1).detach().cpu().numpy()

In [ ]:
# Green: ground truth
imgs = draw_predictions(imgs, classes, bboxes, [(0, 150, 0)], (5, 10))
# Red: predicted
pred_classes_ = []
for i in range(num_imgs):
    temp = np.array([pred_classes[i]])
    pred_classes_.append(temp)
imgs = draw_predictions(imgs, pred_classes_, pred_bboxes, [(200, 0, 0)], (150, 10))

fig, axes = plt.subplots(nrows=nrows, ncols=num_imgs // nrows, figsize=(30, 30))

for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i])
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
!wget https://autoportal.com/cdn/media/cache/image_300x225/img/new-cars-gallery/chevrolet/camaro/photo34/chevrolet-camaro-79ed1401.jpg
!ls

In [ ]:
pickle_rick_img = io.imread('chevrolet-camaro-79ed1401.jpg')
pickle_rick_img = cv2.resize(pickle_rick_img, (h, w))
pickle_rick_img_=np.expand_dims(pickle_rick_img.transpose(2,0,1), 0)

In [ ]:
plt.imshow(pickle_rick_img)

In [ ]:
pickle_rick_img.shape, pickle_rick_img_.shape

In [ ]:
preds = model(torch.tensor(pickle_rick_img_).float().to(device))
pred_classes = preds['class_id'].argmax(-1).detach().cpu().numpy()
pred_bboxes = preds['bbox'].detach().cpu().numpy().tolist()
pred_class = id2obj[pred_classes[0]]
xmin=int(pred_bboxes[0][0]*w)
ymin=int(pred_bboxes[0][1]*h)
xmax=int(pred_bboxes[0][2]*w)
ymax=int(pred_bboxes[0][3]*h)
pickle_rick_img_ = cv2.rectangle(pickle_rick_img, (xmin, ymin), (xmax, ymax), (255, 0, 0))
pickle_rick_img_ = cv2.putText(pickle_rick_img_, pred_class, (xmin, ymin-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1, cv2.LINE_AA)
plt.figure(figsize=(5, 5))
plt.imshow(pickle_rick_img_)
#plt.axis("off")
plt.show()

# Submission

In [ ]:
# ====================================================
# SUBMISSION - ENSEMBLE LOGITS
# bbox  : VGG16        (IoU=0.8349)
# clase : ConvNeXt + ResNet50 (promedio logits antes del argmax)
#         ConvNeXt acc=0.8487 | ResNet50 acc=0.8061
# ====================================================
device = "cuda"

test_root_dir = osp.join(DATA_DIR, "cars/cars_resize")
test_df       = pd.read_csv(osp.join(DATA_DIR, "test.csv"))

test_ds   = CarsDataset(test_df, root_dir=test_root_dir, labeled=False,
                        transform=eval_transforms_imagenet)
test_data = DataLoader(test_ds, batch_size=32, num_workers=cpu_count(), shuffle=False)

bbox_model.to(device).eval()
class_model.to(device).eval()        # ConvNeXt-Tiny
model_resnet50.to(device).eval()     # ResNet50 (segundo mejor acc)

bbox_preds, class_preds = [], []
with torch.no_grad():
    for batch in test_data:
        imgs = batch["image"].float().to(device)
        # Bbox: solo VGG16 (mejor IoU)
        bbox_preds.extend(bbox_model(imgs)["bbox"].cpu().numpy().tolist())
        # Clase: promedio de logits de ConvNeXt y ResNet50
        logits_cnx = class_model(imgs)["class_id"]       # (B, 22)
        logits_rn  = model_resnet50(imgs)["class_id"]    # (B, 22)
        avg_logits = (2.0 * logits_cnx + logits_rn) / 3.0  # ConvNeXt pesa doble (mejor acc)
        class_preds.extend(avg_logits.argmax(-1).cpu().numpy().tolist())

class_preds = np.array(class_preds)
bbox_preds  = np.array(bbox_preds)
print(f"Ensemble logits completado: {len(class_preds)} imagenes")

In [ ]:
# ALTERNATIVA: ConvNeXt solo para bbox Y clase (diagnostico)
# Descomentar para correr solo con ConvNeXt
# model_convnext.to(device).eval()
# class_preds_cnx, bbox_preds_cnx = [], []
# with torch.no_grad():
#     for batch in test_data:
#         imgs = batch["image"].float().to(device)
#         pred = model_convnext(imgs)
#         class_preds_cnx.extend(pred["class_id"].argmax(-1).cpu().numpy().tolist())
#         bbox_preds_cnx.extend(pred["bbox"].cpu().numpy().tolist())
# class_preds = np.array(class_preds_cnx)
# bbox_preds  = np.array(bbox_preds_cnx)
# print(f"ConvNeXt solo: {len(class_preds)} imagenes")

# PARA USAR MODELO DE FINE-TUNING (si se entreno):
# class_model = load_model_from_checkpoint("best_model_convnext_ft.pth",
#                ConvNeXtTinyFeatureExtractor, device, hidden_dim=512, num_classes=22)
print("Opciones de submission - ver comentarios arriba")

In [ ]:
submission = pd.DataFrame(
    index=test_df.filename,
    data={"class": class_preds}
)
submission["xmin"] = bbox_preds[:, 0] * w
submission["ymin"] = bbox_preds[:, 1] * h
submission["xmax"] = bbox_preds[:, 2] * w
submission["ymax"] = bbox_preds[:, 3] * h
submission

In [ ]:
submission['class'] = submission['class'].replace(id2obj)

In [ ]:
submission['class'].value_counts()

In [ ]:
submission

In [ ]:
submission.to_csv('submission(a5).csv')
print("Guardada: submission(a5).csv  (VGG16 bbox + ensemble logits ConvNeXt+ResNet50)")

---
# TTA Submission (Test Time Augmentation)

Mejora la prediccion de clase promediando logits sobre:
- imagen original
- flip horizontal

Bbox sigue usando VGG16 sin TTA (hflip invertido es no trivial y VGG16 ya tiene IoU alto).

In [ ]:
import torchvision.transforms.functional as TF

# ====================================================
# TTA: Ensemble logits (ConvNeXt + ResNet50) x (original + hflip)
# 4 predicciones promediadas por imagen
# bbox: VGG16 sin TTA (ya tiene IoU alto)
# ====================================================
class_model.to(device).eval()      # ConvNeXt
model_resnet50.to(device).eval()   # ResNet50
bbox_model.to(device).eval()       # VGG16

# Reutiliza test_data definido en celda de submission
class_preds_tta, bbox_preds_tta = [], []

with torch.no_grad():
    for batch in test_data:
        imgs      = batch["image"].float().to(device)  # (B, 3, H, W)
        imgs_flip = TF.hflip(imgs)

        # Bbox: VGG16 sobre imagen original
        bbox_preds_tta.extend(bbox_model(imgs)["bbox"].cpu().numpy().tolist())

        # Clase: 4 preds -> promedio logits
        l_cnx_orig  = class_model(imgs)["class_id"]
        l_cnx_flip  = class_model(imgs_flip)["class_id"]
        l_rn_orig   = model_resnet50(imgs)["class_id"]
        l_rn_flip   = model_resnet50(imgs_flip)["class_id"]
        # ConvNeXt pesa doble: (2*cnx_orig + 2*cnx_flip + rn_orig + rn_flip) / 6
        avg_logits  = (2.0*l_cnx_orig + 2.0*l_cnx_flip + l_rn_orig + l_rn_flip) / 6.0
        class_preds_tta.extend(avg_logits.argmax(-1).cpu().numpy().tolist())

class_preds_tta = np.array(class_preds_tta)
bbox_preds_tta  = np.array(bbox_preds_tta)

submission_tta = pd.DataFrame(index=test_df.filename, data={"class": class_preds_tta})
submission_tta["xmin"] = bbox_preds_tta[:, 0] * w
submission_tta["ymin"] = bbox_preds_tta[:, 1] * h
submission_tta["xmax"] = bbox_preds_tta[:, 2] * w
submission_tta["ymax"] = bbox_preds_tta[:, 3] * h
submission_tta["class"] = submission_tta["class"].replace(id2obj)

submission_tta.to_csv("submission_tta_ensemble.csv")
print("TTA ensemble guardada: submission_tta_ensemble.csv  (ConvNeXt+ResNet50 x orig+hflip)")
print(submission_tta["class"].value_counts())
submission_tta